In [ ]:
# Cell 1: Setup for Umbaugh 2h dataset

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from importlib.metadata import version

# Scanpy settings
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=150, facecolor='white', figsize=(8, 6))

# Paths
from pathlib import Path
base_path = os.environ.get("TRAJRISK_ROOT") or str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "metadata" / "source_provenance.json").is_file()))
data_raw = os.path.join(base_path, "Data/raw")
data_processed = os.path.join(base_path, "Data/processed")
results_path = os.path.join(base_path, "results")

print("✅ Setup complete for Umbaugh 2h dataset!")
print(f"Scanpy version: {version('scanpy')}")

In [ ]:
# Cell 2: List Umbaugh 2h H5 files

umbaugh_2h_path = os.path.join(data_raw, "umbaugh_2h_GSE200771")

# List all H5 files
h5_files = [f for f in os.listdir(umbaugh_2h_path) if f.endswith('.h5')]
h5_files.sort()

print("📁 Found H5 files in umbaugh_2h_GSE200771:")
print("="*80)
for f in h5_files:
    file_path = os.path.join(umbaugh_2h_path, f)
    file_size = os.path.getsize(file_path) / 1e6
    print(f"   {f:<50} ({file_size:>6.1f} MB)")
print("="*80)
print(f"\n📊 Total H5 files: {len(h5_files)}")

# Verify we have exactly 6 files (3 control + 3 APAP)
if len(h5_files) == 6:
    print("✅ Correct number of files (6 expected)")
else:
    print(f"⚠️  WARNING: Expected 6 files, found {len(h5_files)}")

In [ ]:
# Cell 3: Load all 6 samples - OPTIMIZED for speed

adatas = []

print("🔄 Loading samples with efficient filtering...\n")

for i, h5_file in enumerate(h5_files, 1):
    file_path = os.path.join(umbaugh_2h_path, h5_file)
    
    # Extract sample ID
    sample_id = h5_file.replace('_raw_feature_bc_matrix.h5', '')
    
    print(f"[{i}/6] Loading {sample_id}...")
    
    # Load H5 file
    adata = sc.read_10x_h5(file_path, gex_only=True)
    
    # Make gene names unique FIRST (before any operations)
    adata.var_names_make_unique()
    
    print(f"      Raw: {adata.n_obs:>8,} barcodes")
    
    # FAST FILTER: Use matrix directly (no QC calculation needed)
    # Count number of genes per cell (non-zero entries per row)
    genes_per_cell = (adata.X > 0).sum(axis=1).A1  # Fast sparse matrix operation
    
    # Keep only cells with >200 genes
    cell_filter = genes_per_cell > 200
    adata = adata[cell_filter, :].copy()
    
    print(f"      Filtered: {adata.n_obs:>6,} cells (>200 genes)")
    
    # Make cell barcodes unique
    adata.obs_names = [f"{barcode}_{sample_id}" for barcode in adata.obs_names]
    
    # Add sample metadata
    adata.obs['sample'] = sample_id
    
    # Store
    adatas.append(adata)
    
    print(f"      ✓ Done\n")

print("="*80)

# Concatenate
print("🔗 Concatenating all samples...")
adata_umbaugh_2h = sc.concat(adatas, join='outer', index_unique=None)

print("\n✅ Umbaugh 2h dataset loaded successfully!\n")
print(f"📊 DATASET SUMMARY:")
print(f"   Total cells: {adata_umbaugh_2h.n_obs:,}")
print(f"   Total genes: {adata_umbaugh_2h.n_vars:,}")
print(f"   Shape: {adata_umbaugh_2h.shape}")

print(f"\n📊 Cells per sample:")
sample_counts = adata_umbaugh_2h.obs['sample'].value_counts().sort_index()
for sample, count in sample_counts.items():
    print(f"   {sample}: {count:>6,} cells")

# Calculate QC annotations before their first use in plots and filtering.
adata_umbaugh_2h.var["mt"] = adata_umbaugh_2h.var_names.str.upper().str.startswith("MT-")
sc.pp.calculate_qc_metrics(
    adata_umbaugh_2h, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True
)


In [ ]:
# Cell 4: Display sample information

print("📋 SAMPLE INFORMATION NEEDED:")
print("="*80)
print("We have 6 samples but need to identify which are Control vs APAP")
print("\nSample IDs:")
for i, sample in enumerate(sorted(adata_umbaugh_2h.obs['sample'].unique()), 1):
    count = (adata_umbaugh_2h.obs['sample'] == sample).sum()
    print(f"  {i}. {sample}: {count:>6,} cells")

print("\n" + "="*80)
print("\n🔍 CHECKING GEO METADATA...")
print("\nFrom GEO GSE200771, the samples should be:")
print("  - 3 Control samples (0h)")
print("  - 3 APAP samples (2h post-injury)")
print("\nWe need to check the GEO page to map sample numbers to conditions.")
print("\nExpected pattern from Umbaugh paper:")
print("  - Samples ending in ...81/...82/...83 = likely one condition")
print("  - Samples ending in ...99/...00/...84 = likely other condition")

In [ ]:
# Cell 4: Add treatment and timepoint metadata

# Define sample-to-treatment mapping based on Umbaugh et al. 2021 paper
# Control (vehicle): n=3, APAP (2h): n=3

treatment_map = {
    'GSM6043450_3381': 'Control',
    'GSM6043452_3183': 'Control',
    'GSM6043455_3184': 'Control',
    'GSM6043451_3399': 'APAP',
    'GSM6043453_3382': 'APAP',
    'GSM6043454_3400': 'APAP'
}

# Add treatment labels
adata_umbaugh_2h.obs['treatment'] = adata_umbaugh_2h.obs['sample'].map(treatment_map)

# Add timepoint (all samples are 2h timepoint)
adata_umbaugh_2h.obs['timepoint'] = '2h'

# Add timepoint_hours for trajectory analysis
adata_umbaugh_2h.obs['timepoint_hours'] = 2

# Verify the mapping
print("📊 TREATMENT DISTRIBUTION:")
print("="*80)
print(adata_umbaugh_2h.obs.groupby(['treatment', 'sample']).size().unstack(fill_value=0))

print("\n📊 SUMMARY:")
treatment_counts = adata_umbaugh_2h.obs['treatment'].value_counts()
for treatment, count in treatment_counts.items():
    print(f"   {treatment}: {count:>6,} cells")

print("\n✅ Metadata added successfully!")

In [ ]:
# Cell 5B: Investigate data quality distribution

import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Plot 1: Genes per cell - full distribution
axes[0, 0].hist(adata_umbaugh_2h.obs['n_genes_by_counts'], bins=100, color='#9b59b6', alpha=0.7)
axes[0, 0].axvline(200, color='red', linestyle='--', linewidth=2, label='Our filter (>200)')
axes[0, 0].axvline(500, color='orange', linestyle='--', linewidth=2, label='Typical min (>500)')
axes[0, 0].set_xlabel('Genes per cell')
axes[0, 0].set_ylabel('Number of cells')
axes[0, 0].set_title('Genes per Cell Distribution')
axes[0, 0].legend()
axes[0, 0].set_yscale('log')

# Plot 2: UMI per cell - full distribution
axes[0, 1].hist(adata_umbaugh_2h.obs['total_counts'], bins=100, color='#3498db', alpha=0.7)
axes[0, 1].axvline(500, color='red', linestyle='--', linewidth=2, label='Very low')
axes[0, 1].axvline(1000, color='orange', linestyle='--', linewidth=2, label='Low')
axes[0, 1].axvline(3000, color='green', linestyle='--', linewidth=2, label='Typical min')
axes[0, 1].set_xlabel('UMI per cell')
axes[0, 1].set_ylabel('Number of cells')
axes[0, 1].set_title('UMI per Cell Distribution')
axes[0, 1].legend()
axes[0, 1].set_yscale('log')

# Plot 3: Mito % distribution
axes[0, 2].hist(adata_umbaugh_2h.obs['pct_counts_mt'], bins=100, color='#e74c3c', alpha=0.7)
axes[0, 2].axvline(5, color='green', linestyle='--', linewidth=2, label='Typical min')
axes[0, 2].axvline(15, color='orange', linestyle='--', linewidth=2, label='Typical max')
axes[0, 2].set_xlabel('Mitochondrial %')
axes[0, 2].set_ylabel('Number of cells')
axes[0, 2].set_title('Mitochondrial % Distribution')
axes[0, 2].legend()

# Plot 4: Genes vs UMI scatter
axes[1, 0].scatter(adata_umbaugh_2h.obs['total_counts'], 
                   adata_umbaugh_2h.obs['n_genes_by_counts'],
                   s=1, alpha=0.3, c='blue')
axes[1, 0].axhline(500, color='red', linestyle='--', alpha=0.5)
axes[1, 0].axvline(3000, color='red', linestyle='--', alpha=0.5)
axes[1, 0].set_xlabel('UMI per cell')
axes[1, 0].set_ylabel('Genes per cell')
axes[1, 0].set_title('Genes vs UMI Relationship')
axes[1, 0].set_xscale('log')
axes[1, 0].set_yscale('log')

# Plot 5: Distribution percentiles
percentiles = [10, 25, 50, 75, 90, 95, 99]
genes_pct = np.percentile(adata_umbaugh_2h.obs['n_genes_by_counts'], percentiles)
umi_pct = np.percentile(adata_umbaugh_2h.obs['total_counts'], percentiles)

axes[1, 1].plot(percentiles, genes_pct, marker='o', label='Genes/cell', linewidth=2)
axes[1, 1].plot(percentiles, umi_pct, marker='s', label='UMI/cell', linewidth=2)
axes[1, 1].axhline(1500, color='red', linestyle='--', alpha=0.5, label='Expected min genes')
axes[1, 1].axhline(3000, color='orange', linestyle='--', alpha=0.5, label='Expected min UMI')
axes[1, 1].set_xlabel('Percentile')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Quality Percentiles')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# Plot 6: Cell counts that pass stricter filters
filter_thresholds = [200, 300, 500, 700, 1000, 1500, 2000]
passing_cells = [
    (adata_umbaugh_2h.obs['n_genes_by_counts'] >= thresh).sum() 
    for thresh in filter_thresholds
]

axes[1, 2].bar(range(len(filter_thresholds)), passing_cells, color='#2ecc71')
axes[1, 2].set_xticks(range(len(filter_thresholds)))
axes[1, 2].set_xticklabels([f'>{t}' for t in filter_thresholds], rotation=45)
axes[1, 2].set_xlabel('Min genes threshold')
axes[1, 2].set_ylabel('Cells passing filter')
axes[1, 2].set_title('Cells Passing Stricter Filters')
for i, count in enumerate(passing_cells):
    axes[1, 2].text(i, count + 5000, f'{count:,}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print("📊 QUALITY PERCENTILES:")
print("="*80)
print(f"{'Percentile':<12} {'Genes/cell':>12} {'UMI/cell':>12}")
print("-"*80)
for pct, genes, umi in zip(percentiles, genes_pct, umi_pct):
    print(f"{pct}th %ile    {genes:>12.0f} {umi:>12.0f}")

print(f"\n📊 CELLS PASSING STRICTER FILTERS:")
print("="*80)
for thresh, count in zip(filter_thresholds, passing_cells):
    pct = 100 * count / adata_umbaugh_2h.n_obs
    print(f"  >{thresh:>4} genes: {count:>8,} cells ({pct:>5.1f}%)")

In [ ]:
# Cell 7: Apply final strict filter (>1000 genes)

print("📊 BEFORE FINAL FILTERING:")
print(f"   Total cells: {adata_umbaugh_2h.n_obs:,}")
print(f"   Median genes/cell: {adata_umbaugh_2h.obs['n_genes_by_counts'].median():.0f}")
print(f"   Median UMI/cell: {adata_umbaugh_2h.obs['total_counts'].median():.0f}")

# Apply final filter: >1000 genes
adata_umbaugh_2h_final = adata_umbaugh_2h[
    adata_umbaugh_2h.obs['n_genes_by_counts'] > 1000, :
].copy()

print(f"\n📊 AFTER FINAL FILTERING (>1000 genes):")
print(f"   Total cells: {adata_umbaugh_2h_final.n_obs:,}")
print(f"   Median genes/cell: {adata_umbaugh_2h_final.obs['n_genes_by_counts'].median():.0f}")
print(f"   Median UMI/cell: {adata_umbaugh_2h_final.obs['total_counts'].median():.0f}")
print(f"   Median mito %: {adata_umbaugh_2h_final.obs['pct_counts_mt'].median():.2f}%")

print(f"\n📊 CELLS REMOVED IN THIS ROUND:")
removed = adata_umbaugh_2h.n_obs - adata_umbaugh_2h_final.n_obs
print(f"   {removed:,} cells removed ({100*removed/adata_umbaugh_2h.n_obs:.1f}%)")

print(f"\n📊 TOTAL CELLS REMOVED FROM ORIGINAL:")
total_removed = 395954 - adata_umbaugh_2h_final.n_obs
print(f"   {total_removed:,} cells removed ({100*total_removed/395954:.1f}%)")
print(f"   Kept: {adata_umbaugh_2h_final.n_obs:,} cells ({100*adata_umbaugh_2h_final.n_obs/395954:.1f}%)")

print(f"\n📊 FINAL TREATMENT DISTRIBUTION:")
for treatment in ['Control', 'APAP']:
    subset = adata_umbaugh_2h_final.obs[adata_umbaugh_2h_final.obs['treatment'] == treatment]
    n_samples = subset['sample'].nunique()
    print(f"\n{treatment}: {len(subset):,} cells ({n_samples} samples)")
    print(f"  Median genes/cell: {subset['n_genes_by_counts'].median():.0f}")
    print(f"  Median UMI/cell: {subset['total_counts'].median():.0f}")
    print(f"  Median mito %: {subset['pct_counts_mt'].median():.2f}%")
    print(f"  Cells per sample:")
    for sample in sorted(subset['sample'].unique()):
        n = (subset['sample'] == sample).sum()
        print(f"    {sample}: {n:,} cells")

# Check if we have enough cells
if adata_umbaugh_2h_final.n_obs < 5000:
    print("\n⚠️  WARNING: Very few cells remaining (<5,000)")
    print("   This dataset may have quality issues.")
elif adata_umbaugh_2h_final.n_obs < 10000:
    print("\n⚠️  Low cell count (<10,000) but usable")
else:
    print("\n✅ Good cell count for analysis")

# Replace main object
adata_umbaugh_2h = adata_umbaugh_2h_final.copy()

print("\n✅ Final filtering complete!")

In [ ]:
# Cell 8: Save final filtered Umbaugh 2h dataset

# Save as AnnData
output_file = os.path.join(data_processed, "umbaugh_2h_filtered.h5ad")
adata_umbaugh_2h.write(output_file)

print(f"✅ Dataset saved to: {output_file}")
print(f"   File size: {os.path.getsize(output_file) / 1e6:.1f} MB")

print(f"\n📊 FINAL SAVED DATASET:")
print("="*80)
print(f"   Total cells: {adata_umbaugh_2h.n_obs:,}")
print(f"   Total genes: {adata_umbaugh_2h.n_vars:,}")
print(f"   Treatments: Control ({(adata_umbaugh_2h.obs['treatment'] == 'Control').sum():,} cells), "
      f"APAP ({(adata_umbaugh_2h.obs['treatment'] == 'APAP').sum():,} cells)")
print(f"   Timepoint: 2h")
print(f"\n   Quality metrics:")
print(f"   - Median genes/cell: {adata_umbaugh_2h.obs['n_genes_by_counts'].median():.0f}")
print(f"   - Median UMI/cell: {adata_umbaugh_2h.obs['total_counts'].median():.0f}")
print(f"   - Median mito %: {adata_umbaugh_2h.obs['pct_counts_mt'].median():.2f}%")
print("="*80)

print("\n✅ Umbaugh 2h dataset saved successfully!")

In [ ]:
# Cell 9: Create comprehensive summary table

summary_data = []

for treatment in ['Control', 'APAP']:
    treatment_data = adata_umbaugh_2h.obs[adata_umbaugh_2h.obs['treatment'] == treatment]
    
    n_cells = len(treatment_data)
    n_samples = treatment_data['sample'].nunique()
    samples = treatment_data['sample'].unique()
    median_genes = treatment_data['n_genes_by_counts'].median()
    median_umi = treatment_data['total_counts'].median()
    median_mito = treatment_data['pct_counts_mt'].median()
    
    summary_data.append({
        'Dataset': 'Umbaugh_2h',
        'Treatment': treatment,
        'Timepoint': '2h',
        'Cells': n_cells,
        'Samples': n_samples,
        'Sample_IDs': ', '.join(sorted(samples)),
        'Median_Genes': int(median_genes),
        'Median_UMI': int(median_umi),
        'Median_Mito_Pct': f"{median_mito:.2f}"
    })

summary_df = pd.DataFrame(summary_data)

print("📋 UMBAUGH 2H FINAL SUMMARY:")
print("="*80)
print(summary_df.to_string(index=False))
print("="*80)

# Save summary
summary_file = os.path.join(data_processed, "umbaugh_2h_summary.csv")
summary_df.to_csv(summary_file, index=False)
print(f"\n✅ Summary saved to: {summary_file}")

In [ ]:
# Cell 10: Create final publication-quality figures

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Plot 1: Cells per treatment
treatment_counts = adata_umbaugh_2h.obs['treatment'].value_counts()
axes[0, 0].bar(treatment_counts.index, treatment_counts.values, color=['#2ecc71', '#e74c3c'], width=0.6)
axes[0, 0].set_ylabel('Number of Cells', fontweight='bold', fontsize=11)
axes[0, 0].set_title('Cells per Treatment', fontweight='bold', fontsize=12)
for i, (idx, val) in enumerate(treatment_counts.items()):
    axes[0, 0].text(i, val + 300, f'{val:,}', ha='center', fontweight='bold', fontsize=10)

# Plot 2: Cells per sample
sample_counts = adata_umbaugh_2h.obs.groupby('sample').size().sort_index()
colors = ['#2ecc71', '#e74c3c', '#2ecc71', '#e74c3c', '#e74c3c', '#2ecc71']
axes[0, 1].bar(range(len(sample_counts)), sample_counts.values, color=colors, width=0.7)
axes[0, 1].set_xticks(range(len(sample_counts)))
axes[0, 1].set_xticklabels([s.split('_')[0] for s in sample_counts.index], rotation=45, ha='right', fontsize=9)
axes[0, 1].set_ylabel('Number of Cells', fontweight='bold', fontsize=11)
axes[0, 1].set_title('Cells per Sample', fontweight='bold', fontsize=12)

# Plot 3: Genes per cell by treatment
control_genes = adata_umbaugh_2h.obs[adata_umbaugh_2h.obs['treatment'] == 'Control']['n_genes_by_counts']
apap_genes = adata_umbaugh_2h.obs[adata_umbaugh_2h.obs['treatment'] == 'APAP']['n_genes_by_counts']
axes[0, 2].violinplot([control_genes, apap_genes], positions=[0, 1], 
                       showmeans=True, showmedians=True, widths=0.7)
axes[0, 2].set_xticks([0, 1])
axes[0, 2].set_xticklabels(['Control', 'APAP'])
axes[0, 2].set_ylabel('Genes per Cell', fontweight='bold', fontsize=11)
axes[0, 2].set_title('Genes per Cell by Treatment', fontweight='bold', fontsize=12)

# Plot 4: UMI per cell by treatment
control_umi = adata_umbaugh_2h.obs[adata_umbaugh_2h.obs['treatment'] == 'Control']['total_counts']
apap_umi = adata_umbaugh_2h.obs[adata_umbaugh_2h.obs['treatment'] == 'APAP']['total_counts']
axes[1, 0].violinplot([control_umi, apap_umi], positions=[0, 1], 
                       showmeans=True, showmedians=True, widths=0.7)
axes[1, 0].set_xticks([0, 1])
axes[1, 0].set_xticklabels(['Control', 'APAP'])
axes[1, 0].set_ylabel('UMI per Cell', fontweight='bold', fontsize=11)
axes[1, 0].set_title('UMI per Cell by Treatment', fontweight='bold', fontsize=12)

# Plot 5: Mito % by treatment
control_mito = adata_umbaugh_2h.obs[adata_umbaugh_2h.obs['treatment'] == 'Control']['pct_counts_mt']
apap_mito = adata_umbaugh_2h.obs[adata_umbaugh_2h.obs['treatment'] == 'APAP']['pct_counts_mt']
axes[1, 1].violinplot([control_mito, apap_mito], positions=[0, 1], 
                       showmeans=True, showmedians=True, widths=0.7)
axes[1, 1].set_xticks([0, 1])
axes[1, 1].set_xticklabels(['Control', 'APAP'])
axes[1, 1].set_ylabel('Mitochondrial %', fontweight='bold', fontsize=11)
axes[1, 1].set_title('Mitochondrial % by Treatment', fontweight='bold', fontsize=12)

# Plot 6: Genes vs UMI scatter colored by treatment
control_data = adata_umbaugh_2h.obs[adata_umbaugh_2h.obs['treatment'] == 'Control']
apap_data = adata_umbaugh_2h.obs[adata_umbaugh_2h.obs['treatment'] == 'APAP']
axes[1, 2].scatter(control_data['total_counts'], control_data['n_genes_by_counts'], 
                   s=3, alpha=0.4, c='#2ecc71', label='Control')
axes[1, 2].scatter(apap_data['total_counts'], apap_data['n_genes_by_counts'], 
                   s=3, alpha=0.4, c='#e74c3c', label='APAP')
axes[1, 2].set_xlabel('UMI per Cell', fontweight='bold', fontsize=11)
axes[1, 2].set_ylabel('Genes per Cell', fontweight='bold', fontsize=11)
axes[1, 2].set_title('Genes vs UMI', fontweight='bold', fontsize=12)
axes[1, 2].legend(frameon=True, loc='lower right')

plt.tight_layout()

# Save figure
fig_path = os.path.join(results_path, 'figures/umbaugh_2h_final_qc.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Figure saved to: {fig_path}")